Comparative Evaluation：Diffusion models vs Autoregressive models


1. Fine tune autoregressive model Llama 3.1 (baseline)
2. Test on test set
3. Metrics
4. LlaDa, LAD
5. Compare

In [1]:
#deep learning
import torch

# Hugging Face
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from huggingface_hub import login
from getpass import getpass

import pandas as pd
import numpy as np

#Metrics
from rouge_score import rouge_scorer
import bert_score

import nltk
from collections import Counter
import string
from wordcloud import WordCloud

import matplotlib.pyplot as plt

import os
import zipfile

In [2]:
base_path = os.getcwd()
data_roots = [
    'multiclinsum_gs_train_en',
    'multiclinsum_large-scale_train_en',
    'multiclinsum_test_en'
]

dfs = {}

for folder in data_roots:
    zip_path = os.path.join(base_path, 'Data', f'{folder}.zip')
    records = []
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Iterate through all files in the zip
        for file in z.namelist():
            if '/fulltext/' in file and file.endswith('.txt'):
                # Read the full text
                full_text = z.read(file).decode('utf-8')
                
                # Build corresponding summary file path
                summary_file = file.replace('/fulltext/', '/summaries/').replace('.txt', '_sum.txt')
                summary_text = z.read(summary_file).decode('utf-8')
                
                records.append({'Full_Text': full_text, 'Summary': summary_text})
    
    dfs[folder] = pd.DataFrame(records)

df_gs = dfs['multiclinsum_gs_train_en']
df_large = dfs['multiclinsum_large-scale_train_en']
df_test = dfs['multiclinsum_test_en']

In [3]:
#Prepare training data
def format_example(row):
    return {"text": f"Summarize this clinical note: {row['Full_Text']}\nSummary: {row['Summary']}"}

train_data = [format_example(row) for _, row in df_gs.iterrows()]
dataset = Dataset.from_list(train_data)

In [4]:
hf_token = getpass("Enter your Hugging Face token: ")

model_name = "meta-llama/Meta-Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

C:\Users\leeze\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\leeze\.cache\huggingface\hub\models--meta-llama--Meta-Llama-3.1-8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.layers.14.self_attn.q_proj.weight to model.layers.14.self_attn.k_proj.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.layers.14.self_attn.q_proj.weight to model.layers.14.self_attn.v_proj.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.layers.14.self_attn.q_proj.weight to model.layers.14.self_attn.o_proj.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.layers.14.self

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [5]:
#Lora
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

In [6]:
#tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/592 [00:00<?, ? examples/s]

In [7]:
# Training parameters
training_args = TrainingArguments(
    output_dir="./llama-clinical-summary",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=500,
    bf16=True,
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


: 

In [ ]:
model.save_pretrained("./llama-clinical-summary-lora")
tokenizer.save_pretrained("./llama-clinical-summary-lora")